<a href="https://colab.research.google.com/github/adeeljames/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/adeeljames/flyrank-ml-internship"
REPO_DIR = "flyrank-ml-internship"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)

import pandas as pd, numpy as np
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = df["trend_direction"].str.lower().eq("down").astype(int)
print(df.shape)

(30000, 45)


## 1. Two paper findings + my methodology questions

**Finding 1: "The Content Performance Curve" (Finding #2) — content peaks at
61-90 days, decays after 270 days, recovers at 365+ if refreshed.**

My methodology question: This health-score-by-age table looks cross-sectional
(different pages of different ages compared at one point in time), not
longitudinal (the same pages tracked as they age). If it's cross-sectional,
older pages that are still active today may be a biased sample — weaker old
pages may have already been pruned, deindexed, or replaced, so the pages that
survive into the 365+ bucket aren't a random sample of "what happens to a
page as it ages." The paper itself flags this exact risk for the smallest
cell in the age-freshness matrix ("the active-content subset introduces
strong survivor bias there"), so my question is simply: does the same
survivorship caveat apply to the main lifecycle curve, not just the smallest
matrix cell?

**Finding 2: ML Appendix — "What Predicts Health?" (Random Forest feature
importance, Average Position 43%, Impressions 32%).**

My methodology question: the paper is admirably transparent that "the target
itself is partly constructed from some of these inputs" — Health Score is
literally defined as Impressions (30 pts) + Position (30 pts) + CTR (20 pts)
+ Scroll Depth (20 pts), and two of those same components (Average Position,
Impressions) are the model's top two "predictors." Since the label is a
linear combination of the features, isn't this closer to the model
re-deriving its own formula than discovering a real predictive relationship?
This is exactly the leakage pattern from my own Week-3 data contract exercise
(a feature that is definitionally part of the label).

(Framed respectfully: both of these are caveats the paper itself gestures
toward but doesn't fully spell out — the same standard of transparency I'm
applying to my own Week-5/Week-6 work below.)

## 2. My model under an honest split (before/after)

Re-running my Week-5 Random Forest model two ways: a naive random row split
(what an unaware analysis might do) versus my client-grouped holdout split
(the honest version, matching Week 5).

In [2]:
from sklearn.model_selection import GroupShuffleSplit, train_test_split
from sklearn.ensemble import RandomForestClassifier

features = ["search_volume", "word_count", "impressions_90d", "sessions_90d",
            "content_age_days", "days_since_last_update", "avg_position",
            "ctr", "engagement_rate", "scroll_rate"]
features = [f for f in features if f in df.columns]

X = df[features].replace([np.inf, -np.inf], np.nan).fillna(0)
y = df["is_declining_label"].values

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

# --- BEFORE: naive random row split ---
X_tr_naive, X_te_naive, y_tr_naive, y_te_naive = train_test_split(
    X, y, test_size=0.2, random_state=42)

rf_naive = RandomForestClassifier(n_estimators=200, max_depth=8, class_weight="balanced", random_state=42)
rf_naive.fit(X_tr_naive, y_tr_naive)
naive_scores = rf_naive.predict_proba(X_te_naive)[:, 1]
naive_p50 = precision_at_k(naive_scores, y_te_naive, min(50, len(y_te_naive)))

# --- AFTER: honest client-grouped holdout ---
groups = df["client_id"].values
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
train_idx, test_idx = next(gss.split(X, y, groups=groups))
X_tr_grp, X_te_grp = X.iloc[train_idx], X.iloc[test_idx]
y_tr_grp, y_te_grp = y[train_idx], y[test_idx]

rf_grp = RandomForestClassifier(n_estimators=200, max_depth=8, class_weight="balanced", random_state=42)
rf_grp.fit(X_tr_grp, y_tr_grp)
grp_scores = rf_grp.predict_proba(X_te_grp)[:, 1]
grp_p50 = precision_at_k(grp_scores, y_te_grp, min(50, len(y_te_grp)))

print(f"BEFORE (naive random split)   Precision@50: {naive_p50:.3f}")
print(f"AFTER  (client-grouped split) Precision@50: {grp_p50:.3f}")
print(f"Difference: {naive_p50 - grp_p50:+.3f}")

BEFORE (naive random split)   Precision@50: 0.900
AFTER  (client-grouped split) Precision@50: 0.640
Difference: +0.260


Before/after result: the naive random split scored {0.900} while the
honest client-grouped split scored {0.640} on Precision@50.

[If naive > grouped:] This gap shows the naive split was optimistic — some
client-specific pattern was leaking between train and test when rows (not
clients) were split randomly. The client-grouped number is the one I trust
and report, matching my Week-5 result.

[If they're close/grouped >= naive:] In this case the two splits produced
similar results, suggesting client-level leakage wasn't a major factor for
this feature set — but I still prefer the grouped split as the more
defensible design going forward, since it removes a plausible leakage path
even where it didn't change the outcome much this time.

## 3. Leakage audit

In [3]:
# 1. Are any features calculated after the decision point / from the label?
print("Features used:", features)
print("trend_direction / trend_pct in features?",
      any(f in ["trend_direction", "trend_pct"] for f in features))

# 2. Confirm label-source columns are excluded
assert "trend_direction" not in features and "trend_pct" not in features
print("Confirmed: label-source columns are excluded from features.")

# 3. Client overlap check between train/test (grouped split)
overlap = set(df["client_id"].iloc[train_idx]) & set(df["client_id"].iloc[test_idx])
print("Client overlap in grouped split:", len(overlap), "(should be 0)")

Features used: ['search_volume', 'word_count', 'impressions_90d', 'sessions_90d', 'content_age_days', 'days_since_last_update', 'avg_position', 'ctr', 'engagement_rate', 'scroll_rate']
trend_direction / trend_pct in features? False
Confirmed: label-source columns are excluded from features.
Client overlap in grouped split: 0 (should be 0)


Leakage audit summary:
- No product decision flags (health_score, priority_score, action_type) are
  used — they were never shipped in this dataset by design.
- trend_direction and trend_pct (the label source) are explicitly excluded
  from the feature list.
- No client appears in both train and test under the grouped split (0 overlap
  confirmed above).
- All features (impressions, position, CTR, staleness, etc.) are observable,
  pre-decision signals — none are calculated from a future window.
- Applying the same standard I used on the FlyRank paper in Section 1: unlike
  the paper's Health Score / Random Forest appendix, my label
  (is_declining_label) is NOT a linear combination of my model's input
  features, so this specific circularity risk does not apply to my own model.

## 4. Claim rewrite

**Original (Week 5) claim, as first written:** "Random Forest beats my
baseline, so it's a better model for this lane."

**Rewritten, safer version:** "Under a client-grouped holdout split, Random
Forest achieved a modestly higher Precision@50 (0.640) than my Week-4
baseline rule (0.620), and better overall ranking quality (ROC AUC 0.598 vs
Logistic Regression's 0.530). This is an observed, directional result on this
specific starter slice — not proof that Random Forest will outperform on the
full warehouse data, and not proof that any specific page will actually
improve if refreshed. The result should be read as decision-support evidence,
not a guarantee."

**Original claim about staleness (Week 4):** "Stale pages are declining
pages."

**Rewritten:** "Staleness alone was only a MIXED signal in my Week-4 audit —
decline rate rose from <90 days to 90-180 days, but did not hold consistently
beyond that (169 and 5 rows in the older buckets, too small to trust).
Staleness should be treated as one contributing signal among several, not a
standalone predictor."

**Original claim about CTR (Week 4):** "CTR drops as position gets worse."

**Rewritten:** "CTR declined monotonically from 0.355 (page_1) to 0.055
(deep) across every position tier in my Week-4 audit — this was CONFIRMED,
with reasonably large sample sizes in every bucket. Any CTR comparison in my
model or baseline must be done within the same position tier, never across
tiers."

## 5. Self-check

- [x] Named two paper findings with a constructive methodology question each
- [x] Re-ran my Week-5 model under an honest (client-grouped) split, showing
      before/after vs a naive random split
- [x] Completed a leakage audit (features, label-source exclusion, no client
      overlap, and checked my own model against the same circularity risk
      identified in the paper)
- [x] Rewrote three of my own earlier claims into safe, observed/directional
      language